# 🎭 Sarcasm Detector
### DeBERTa-v3-large · Fine-tuned on Reddit Balanced Sarcasm Dataset

---

| | |
|---|---|
| **Model** | `microsoft/deberta-v3-large` (~435M params) |
| **Dataset** | Reddit Balanced Sarcasm (~1M comments) |
| **Input** | comment + optional parent comment |
| **Output** | Sarcastic 😏 / Not Sarcastic 🙂 + confidence |
| **Expected accuracy** | ~87–90% |

---

### 📋 How to use
1. **Runtime → Change runtime type → T4 GPU** (required)
2. Run **Cell 1** to install dependencies
3. Run **Cell 2** to mount Google Drive (where your checkpoint is stored)
4. Run **Cells 3–4** to load the model
5. Run **Cell 5** for the interactive demo UI

> ⚠️ Your checkpoint folder must be uploaded to Google Drive first.  
> Update `GDRIVE_CKPT_PATH` in Cell 3 to match your Drive path.

In [1]:
# @title ⚙️ Cell 1 — Install dependencies { display-mode: "form" }
# @markdown Run this first. It installs the exact same library versions used during training.

import subprocess, sys

print('📦 Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.44.2',
                'tokenizers==0.19.1',
                'sentencepiece==0.1.99',
                'protobuf==3.20.3',
                'accelerate',
                'ipywidgets'],
               check=True)

import torch
print(f'\n✅ Done!')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {"🟢 GPU — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "🔴 CPU (switch to GPU runtime for speed)"}')

📦 Installing packages...

✅ Done!
   PyTorch  : 2.10.0+cu128
   Device   : 🟢 GPU — Tesla T4


In [3]:
# @title 📂 Cell 2 — Mount Google Drive { display-mode: "form" }
# @markdown Your checkpoint folder should be somewhere inside your Drive.
# @markdown A browser popup will ask you to sign in and grant access.

from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')
print('   Browse your files in the 📁 panel on the left to find your checkpoint path.')

Mounted at /content/drive
✅ Google Drive mounted at /content/drive
   Browse your files in the 📁 panel on the left to find your checkpoint path.


In [4]:
# @title 🗂️ Cell 3 — Configure checkpoint path { display-mode: "form" }

# @markdown ### 👇 Update this to your checkpoint location in Google Drive
GDRIVE_CKPT_PATH = '/content/drive/MyDrive/DS 440/Apr12/deberta-large-sarcasm-final'  # @param {type:"string"}

MAX_LEN = 256  # must match what was used during training

import os
if not os.path.isdir(GDRIVE_CKPT_PATH):
    print(f'❌ Path not found: {GDRIVE_CKPT_PATH}')
    print('   Please update GDRIVE_CKPT_PATH above to the correct folder.')
else:
    files = os.listdir(GDRIVE_CKPT_PATH)
    print(f'✅ Checkpoint folder found: {GDRIVE_CKPT_PATH}')
    print(f'   Contents: {files}')

✅ Checkpoint folder found: /content/drive/MyDrive/DS 440/Apr12/deberta-large-sarcasm-final
   Contents: ['config.json', 'model.safetensors', 'training_args.bin', 'added_tokens.json', 'tokenizer_config.json', 'spm.model', 'special_tokens_map.json']


In [5]:
# @title 🤖 Cell 4 — Load model & tokenizer { display-mode: "form" }
# @markdown This loads your fine-tuned checkpoint. Takes ~30s on a T4 GPU.

import torch
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading tokenizer...')
tokenizer = DebertaV2Tokenizer.from_pretrained(GDRIVE_CKPT_PATH)

print('Loading model... (this may take ~30s)')
model = DebertaV2ForSequenceClassification.from_pretrained(GDRIVE_CKPT_PATH)
model.to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ Model ready!')
print(f'   Parameters : {total_params:,}')
print(f'   Device     : {DEVICE.upper()}')


# ── Core prediction function ───────────────────────────────────────────────
def predict_sarcasm(comment: str, parent_comment: str = '') -> dict:
    text = comment.strip()
    if parent_comment.strip():
        text = text + ' [SEP] ' + parent_comment.strip()

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length',
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs      = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    pred_label = int(torch.argmax(logits, dim=-1).item())

    return {
        'label':      pred_label,
        'prob_not':   probs[0],
        'prob_sarc':  probs[1],
    }

print('   predict_sarcasm() is ready ✓')

Loading tokenizer...
Loading model... (this may take ~30s)

✅ Model ready!
   Parameters : 435,063,810
   Device     : CUDA
   predict_sarcasm() is ready ✓


In [15]:
# @title 🎮 Cell 5 — Interactive Demo { display-mode: "form" }
# @markdown Type a comment below and click **Analyze** to detect sarcasm.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Widgets ────────────────────────────────────────────────────────────────
style = {'description_width': '140px'}
layout_wide = widgets.Layout(width='100%')
layout_btn  = widgets.Layout(width='160px', height='36px')

w_comment = widgets.Textarea(
    placeholder='e.g. Oh sure, because that always works out SO well.',
    description='💬 Comment:',
    layout=widgets.Layout(width='100%', height='80px'),
    style=style,
)
w_parent = widgets.Textarea(
    placeholder='(Optional) e.g. Just ignore the haters.',
    description='↩️ Parent comment:',
    layout=widgets.Layout(width='100%', height='60px'),
    style=style,
)
w_button  = widgets.Button(description='  Analyze 🔍', button_style='primary', layout=layout_btn)
w_clear   = widgets.Button(description='  Clear', button_style='', layout=widgets.Layout(width='90px', height='36px'))
w_output  = widgets.Output()

# ── Result renderer ────────────────────────────────────────────────────────
def render_result(comment, parent, result):
    label     = result['label']     # 0 or 1
    prob_not  = result['prob_not']  # float 0-1
    prob_sarc = result['prob_sarc'] # float 0-1

    is_sarc   = label == 1
    emoji     = '😏' if is_sarc else '🙂'
    verdict   = 'Sarcastic' if is_sarc else 'Not Sarcastic'
    conf      = prob_sarc if is_sarc else prob_not
    bar_color = '#e74c3c' if is_sarc else '#2ecc71'
    bg_color  = '#fff5f5' if is_sarc else '#f5fff8'
    border    = '#e74c3c' if is_sarc else '#2ecc71'

    sarc_bar_w = int(prob_sarc * 100)
    not_bar_w  = int(prob_not  * 100)

    parent_html = ''
    if parent.strip():
        parent_html = f'''
        <div style="margin-bottom:8px;">
          <span style="color:#888;font-size:12px;">↩️ PARENT COMMENT</span><br>
          <span style="color:#555;font-style:italic;">"{parent.strip()}"</span>
        </div>'''

    html = f'''
    <div style="font-family: 'Google Sans', sans-serif; max-width:620px; margin:12px 0;">

      <!-- Input card -->
      <div style="background:#f8f9fa; border-radius:10px; padding:14px 18px; margin-bottom:12px; border:1px solid #e0e0e0;">
        {parent_html}
        <div>
          <span style="color:#888;font-size:12px;">💬 COMMENT</span><br>
          <span style="color:#222;font-weight:500;">"{comment.strip()}"</span>
        </div>
      </div>

      <!-- Verdict card -->
      <div style="background:{bg_color}; border:2px solid {border}; border-radius:10px; padding:16px 20px; margin-bottom:12px;">
        <div style="font-size:32px; margin-bottom:4px;">{emoji}</div>
        <div style="font-size:22px; font-weight:700; color:{bar_color};">{verdict}</div>
        <div style="color:#555; font-size:14px;">Confidence: <strong>{conf*100:.1f}%</strong></div>
      </div>

      <!-- Probability bars -->
      <div style="background:#fff; border:1px solid #e0e0e0; border-radius:10px; padding:14px 18px;">
        <div style="font-size:13px; color:#888; margin-bottom:10px;">PROBABILITY BREAKDOWN</div>

        <div style="margin-bottom:10px;">
          <div style="display:flex; justify-content:space-between; font-size:13px; margin-bottom:4px;">
            <span>🙂 Not Sarcastic</span><span><strong>{prob_not*100:.1f}%</strong></span>
          </div>
          <div style="background:#e9ecef; border-radius:6px; height:10px; overflow:hidden;">
            <div style="background:#2ecc71; width:{not_bar_w}%; height:100%; border-radius:6px; transition:width 0.4s;"></div>
          </div>
        </div>

        <div>
          <div style="display:flex; justify-content:space-between; font-size:13px; margin-bottom:4px;">
            <span>😏 Sarcastic</span><span><strong>{prob_sarc*100:.1f}%</strong></span>
          </div>
          <div style="background:#e9ecef; border-radius:6px; height:10px; overflow:hidden;">
            <div style="background:#e74c3c; width:{sarc_bar_w}%; height:100%; border-radius:6px; transition:width 0.4s;"></div>
          </div>
        </div>
      </div>

    </div>
    '''
    display(HTML(html))


# ── Button handlers ────────────────────────────────────────────────────────
def on_analyze(b):
    with w_output:
        clear_output(wait=True)
        comment = w_comment.value.strip()
        parent  = w_parent.value.strip()
        if not comment:
            display(HTML('<p style="color:#e74c3c;">⚠️ Please enter a comment first.</p>'))
            return
        display(HTML('<p style="color:#888;">⏳ Analyzing...</p>'))
        result = predict_sarcasm(comment, parent)
        clear_output(wait=True)
        render_result(comment, parent, result)

def on_clear(b):
    w_comment.value = ''
    w_parent.value  = ''
    with w_output:
        clear_output()

w_button.on_click(on_analyze)
w_clear.on_click(on_clear)

# ── Layout ────────────────────────────────────────────────────────────────
display(HTML('<h3 style="font-family:sans-serif; margin-bottom:4px;">🎭 Sarcasm Detector</h3>'))
display(HTML('<p style="color:#888; font-size:13px; margin-top:0;">Powered by DeBERTa-v3-large · Type a comment and click Analyze</p>'))
display(w_comment)
display(w_parent)
display(widgets.HBox([w_button, w_clear], layout=widgets.Layout(gap='10px', margin='8px 0')))
display(w_output)

Textarea(value='', description='💬 Comment:', layout=Layout(height='80px', width='100%'), placeholder='e.g. Oh …

Textarea(value='', description='↩️ Parent comment:', layout=Layout(height='60px', width='100%'), placeholder='…

Output()

In [17]:
# @title 📚 Cell 6 — Run example predictions { display-mode: "form" }
# @markdown Sanity-check the model against a few hand-picked examples.

from IPython.display import display, HTML

examples = [
    ('Oh yeah, because that always works out SO well.',  'Just ignore them and they will stop bullying you.'),
    ('Thanks for the help, I really appreciate it!',     'Sorry, I cannot assist with that.'),
    ('Wow, what a great idea. Truly groundbreaking.',    'I think we should just do nothing and wait.'),
    ('This is genuinely one of the best meals I ever had.', ''),
    ('Sure, I love working 80-hour weeks for minimum wage.', 'You should be grateful you even have a job.'),
]

rows = ''
for comment, parent in examples:
    r         = predict_sarcasm(comment, parent)
    is_sarc   = r['label'] == 1
    emoji     = '😏' if is_sarc else '🙂'
    verdict   = 'Sarcastic' if is_sarc else 'Not Sarcastic'
    conf      = r['prob_sarc'] if is_sarc else r['prob_not']
    color     = '#e74c3c' if is_sarc else '#27ae60'
    p_html    = f'<br><span style="color:#aaa;font-size:11px;">↩ {parent}</span>' if parent else ''
    rows += f'''
    <tr>
      <td style="padding:10px 14px; font-size:13px; border-bottom:1px solid #f0f0f0;">
        {comment}{p_html}
      </td>
      <td style="padding:10px 14px; text-align:center; font-weight:700; color:{color}; border-bottom:1px solid #f0f0f0; white-space:nowrap;">
        {emoji} {verdict}
      </td>
      <td style="padding:10px 14px; text-align:right; color:#555; border-bottom:1px solid #f0f0f0;">
        {conf*100:.1f}%
      </td>
    </tr>'''

html = f'''
<div style="font-family:sans-serif; max-width:680px;">
  <h4 style="margin-bottom:8px;">📚 Example Predictions</h4>
  <table style="width:100%; border-collapse:collapse; background:#fff; border-radius:10px; overflow:hidden; box-shadow:0 1px 4px rgba(0,0,0,.08);">
    <thead>
      <tr style="background:#f8f9fa;">
        <th style="padding:10px 14px; text-align:left; font-size:12px; color:#888;">COMMENT</th>
        <th style="padding:10px 14px; text-align:center; font-size:12px; color:#888;">PREDICTION</th>
        <th style="padding:10px 14px; text-align:right; font-size:12px; color:#888;">CONFIDENCE</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
'''
display(HTML(html))

COMMENT,PREDICTION,CONFIDENCE
"Oh yeah, because that always works out SO well.↩ Just ignore them and they will stop bullying you.",😏 Sarcastic,98.3%
"Thanks for the help, I really appreciate it!↩ Sorry, I cannot assist with that.",🙂 Not Sarcastic,60.8%
"Wow, what a great idea. Truly groundbreaking.↩ I think we should just do nothing and wait.",😏 Sarcastic,96.5%
This is genuinely one of the best meals I ever had.,🙂 Not Sarcastic,99.4%
"Sure, I love working 80-hour weeks for minimum wage.↩ You should be grateful you even have a job.",😏 Sarcastic,97.8%
